kernel : Python (Pixi)

In [1]:
%load_ext autoreload
%autoreload 2
import gc
import os
import anndata as ad
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from pipeline.utils.get_ensembl_to_symbol import get_ensg_to_symbol
from pipeline.utils.env import find_env_dir
from pipeline.utils.fix_nullable_strings import fix_nullable_strings

SERIES = "ibd"
HUMAN_SAVE_FILE= f"{SERIES}_human_crohn_raw.h5ad"
MOUSE_SAVE_FILE= f"{SERIES}_mouse_raw.h5ad"

pre_h5ad_dir = find_env_dir("PRE_H5AD")

krzak_adata = sc.read_h5ad(os.path.join(pre_h5ad_dir, "krzak_raw.h5ad"))

In [2]:
ensg_to_symbol = get_ensg_to_symbol(version="GRCh37")

vn = pd.Series(krzak_adata.var_names, index=krzak_adata.var_names).astype("string")
is_ensg = vn.str.startswith("ENSG")

ensg_only = vn[is_ensg].str.split(".", n=1).str[0]
mapped_ensg = ensg_only.map(ensg_to_symbol).replace("", pd.NA).fillna(vn[is_ensg])

mapped_all = vn.copy()
mapped_all[is_ensg] = mapped_ensg

krzak_adata.var_names = mapped_all.values.astype(str).tolist() 
krzak_adata.var_names_make_unique()

temp_krzak_file = os.path.join(pre_h5ad_dir, "krzak_raw_symbol.h5ad")
krzak_adata.write_h5ad(temp_krzak_file)

del krzak_adata
gc.collect()

37

In [3]:
ad.experimental.concat_on_disk(
    {
        "kong": os.path.join(pre_h5ad_dir, "kong_raw.h5ad"),
        "krzak": temp_krzak_file
    },
    os.path.join(pre_h5ad_dir, HUMAN_SAVE_FILE),
    join="inner",
    index_unique="-",
    max_loaded_elems=10_000_000_000,
)

In [4]:
human_adata = sc.read_h5ad(os.path.join(pre_h5ad_dir, HUMAN_SAVE_FILE))
assert isinstance(human_adata.X, csr_matrix)
human_adata.obs["sample"] = (human_adata.obs["sample"].astype(str) + "_" + human_adata.obs["series"].astype(str)).astype("category") #type: ignore

human_adata.obs

,celltype_coarse,organ,disease,sample,condition,donor,celltype,series
N105446_L-GTGTGGCTCCGTCAAA-kong,stromal,colon,Crohn's disease,N105446_L_kong,NonI,105446,Fibroblasts ADAMDEC1,kong
N105446_L-CAATACGAGTCCCTAA-kong,stromal,colon,Crohn's disease,N105446_L_kong,NonI,105446,Endothelial cells CD36,kong
N105446_L-CCCTGATAGTGTTCCA-kong,stromal,colon,Crohn's disease,N105446_L_kong,NonI,105446,Fibroblasts ADAMDEC1,kong
N105446_L-CATTGTTAGAGCCCAA-kong,stromal,colon,Crohn's disease,N105446_L_kong,NonI,105446,Fibroblasts ADAMDEC1,kong
N105446_L-TCCATGCGTTCGTTCC-kong,stromal,colon,Crohn's disease,N105446_L_kong,NonI,105446,Fibroblasts KCNN3 LY6H,kong
...,...,...,...,...,...,...,...,...
CTGTGGGGTCTGTAAC-1-scrnacdb9997594__37295__cellranger720_count_37295_scrnacdb9997594_GRCh38-3_0_0-krzak,Plasma B cell,terminal ileum,CD,scrnacdb9997594_krzak,inflamed,IBDverse-1-FB0463BF47ACA3,Plasma B cell XBP1/CD38+,krzak
AGGCCACTCTCACGAA-1-scrnacdb9997594__37295__cellranger720_count_37295_scrnacdb9997594_GRCh38-3_0_0-krzak,Myeloid,terminal ileum,CD,scrnacdb9997594_krzak,inflamed,IBDverse-1-FB0463BF47ACA3,Monocytes S100A8/9+,krzak
TTCCACGGTGTGTCCG-1-scrnacdb9997594__37295__cellranger720_count_37295_scrnacdb9997594_GRCh38-3_0_0-krzak,Myeloid,terminal ileum,CD,scrnacdb9997594_krzak,inflamed,IBDverse-1-FB0463BF47ACA3,Monocytes S100A8/9+,krzak
ATGAAAGGTCTGTGCG-1-scrnacdb9997594__37295__cellranger720_count_37295_scrnacdb9997594_GRCh38-3_0_0-krzak,Myeloid,terminal ileum,CD,scrnacdb9997594_krzak,inflamed,IBDverse-1-FB0463BF47ACA3,Mast cells,krzak


In [6]:
human_adata_clean = human_adata[human_adata.obs.notna().all(axis=1), :] #type: ignore

In [ ]:
condition_map = {
    "uninflamed": "uninflamed",
    "NonI": "uninflamed",
    "inflamed": "inflamed",
    "Heal": "uninflamed",
    "Infl": "inflamed",
}
human_adata_clean.obs["condition"] = human_adata_clean.obs["condition"].map(condition_map)

In [ ]:
disease_map = {
    "Healthy": "normal",
    "CD": "CD",
    "Crohn's disease": "CD",
    "normal": "normal",
}
human_adata_clean.obs["disease"] = human_adata_clean.obs["disease"].map(disease_map)

In [ ]:
human_adata_clean = fix_nullable_strings(human_adata_clean)
human_adata_clean.write_h5ad(os.path.join(pre_h5ad_dir, HUMAN_SAVE_FILE), compression="gzip")